<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Background Subtraction</b></h1>
</div>

This notebook executes the neurointerventional background-subtraction study, reproducing the baseline, evaluating controlled ablations, selecting the retained pipeline, computing sequence-level metrics, and validating all final outputs.


## Setup — Environment and Configuration

Shared imports, helper functions, deterministic settings, and repository-relative paths used by Tasks 1–13.

In [ ]:
import os
import numpy as np
#from scipy import misc
import skimage.io as io
from scipy import ndimage
from skimage.filters import gaussian
from skimage.filters import threshold_otsu
from skimage.morphology import erosion, dilation, opening, closing
from skimage.morphology import disk
from skimage.filters import threshold_otsu
from skimage import color
import fnmatch
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib as mpl

def linear (source, a, b):
    out = a*source+b
    return out


def saturate(source, perinf, persup):
    if perinf == 0:
        minper = np.min(source)
    else:
        minper=stats.scoreatpercentile(source,perinf)

    if persup == 100:
        maxper = np.max(source)
    else:
        maxper=stats.scoreatpercentile(source,persup)

    out = source
    out = (source-minper)/(maxper-minper)
    out[np.where(out>1)]=1.0
    out[np.where(out<0)]=0
    return out

def quadratic(source):
    out = (source)**2
    return out

def mse(im1,im2):
    return np.mean( (im1 - im2) ** 2 )

def sad(im1,im2):
    return np.mean(np.abs(im1 - im2))

def psnr(mse, pixel_max=255): #or 1.0
    return 20 * np.log10(pixel_max/ np.sqrt(mse))

def compute_errors(im1,im2):
    im1*=255
    im2*=255
    return sad(im1,im2), mse(im1,im2), psnr(mse(im1,im2))
from sklearn.mixture import GaussianMixture


## 1. Validate Data and Ground-Truth Paths

Validate the supplied fluoroscopy frames and paired GuideWire/MicroCath annotations before any processing.

In [ ]:
IMDIR = "../data/catheter"
print("IMDIR:", IMDIR)

In [ ]:
from pathlib import Path

data_dir = Path(IMDIR)
frame_numbers = list(range(201, 292, 10))

required_files = []
for frame_id in frame_numbers:
    required_files.extend([
        data_dir / f"frame_{frame_id}.png",
        data_dir / f"{frame_id}_GuideWire.tiff",
        data_dir / f"{frame_id}_MicroCath.tiff",
    ])

missing_files = [str(p) for p in required_files if not p.exists()]
assert not missing_files, f"Missing required files: {missing_files}"

print(f"Validated {len(frame_numbers)} frames and {2 * len(frame_numbers)} annotation files.")


## 2. Reproduce the Original Baseline Pipeline

Reproduce the original first-frame-reference pipeline unchanged to establish the quantitative baseline.

In [ ]:
error_dic = {'sad':[],'mse':[],'psnr':[]}

for num_f in range(201,292,10):

    # Build filenames from number sequence
    im_filename = ''.join(['frame_', str(num_f), '.png'])
    guidewire_filename = ''.join([str(num_f),'_GuideWire.tiff'])
    microcath_filename = ''.join([str(num_f),'_MicroCath.tiff'])

    im_f = os.path.join(IMDIR, im_filename)
    guidewire_f = os.path.join(IMDIR, guidewire_filename)
    microcath_f = os.path.join(IMDIR, microcath_filename)

    print(im_f)

    # Read image
    image = io.imread(im_f,as_gray=True)

    # Read guidewire
    guidewire_mask = io.imread(guidewire_f)
    if len(guidewire_mask.shape)>2:
        guidewire_mask = guidewire_mask[:,:,0]

    # Read microcatheter
    microcath_mask = io.imread(microcath_f)
    if len(microcath_mask.shape)>2:
        microcath_mask = microcath_mask[:,:,0]

    # Build a global mask from the guidewire and the microcatheter
    gt_mask = guidewire_mask
    gt_mask[np.where(microcath_mask<128)]=0

    #dilate mask to account for manual error
    selem = disk(2)
    gt_mask = erosion(gt_mask, selem)

    #make image and mask float to facilitate operations
    image = image.astype(float)/255.
    gt_mask = gt_mask.astype(float)/255.

    #filter and enhance input contrast
    im = image.copy()
    im = gaussian(im,sigma=1.0) #clearly improves SSD with sigma = 1.0
    #im = linear(im,3,0.1)
    im = saturate(im,0,90)

    #use first image as background model ->TODO: change to use more or sliding window
    if num_f == 201:
        im_ref = im.copy()
        continue
    else:

        #morphological filter on image difference to obtain mask
        im = -(im - im_ref)
        im_gauss_diff = im.copy() #Store to display later

        selem = disk(2)
        im = dilation(im, selem)

        #histogram transformation
        im = saturate(im,10,100)

        #thresholding
        im = im > 0.1
        #thresh = threshold_otsu(im) #Otsu does not work well for the first images, they seem to need further denoising
        #im = im > thresh

        #morphological
        selem = disk(2)
        im = opening(im, selem)
        im = 1-im
        im_mask = im.copy()

        #compute errors
        sad_i,mse_i,psnr_i = compute_errors(gt_mask,im_mask)
        error_dic['sad'].append(sad_i)
        error_dic['mse'].append(mse_i)
        error_dic['psnr'].append(psnr(mse_i))

        #Create 3 channel image for overlayed display
        im_show = color.gray2rgb(image)
        r_gt,c_gt = np.where(gt_mask<0.5)
        r_hat,c_hat = np.where(im_mask<0.5)
        im_show[r_gt,c_gt,1]=1 #gt
        im_show[r_hat,c_hat,0]=1 #estimate


    #Display results
    fig=plt.figure(figsize=(12,5))

    plt.subplot(1,5,1,adjustable='box')
    plt.imshow(image, cmap='gray')
    plt.title('original image')

    plt.subplot(1,5,2,adjustable='box')
    plt.imshow(im_gauss_diff,cmap='gray')
    plt.title('intermediate diff')

    plt.subplot(1,5,3)
    plt.imshow(im_mask, cmap='gray')
    plt.title('my mask')

    plt.subplot(1,5,4,adjustable='box')
    plt.imshow(gt_mask, cmap='gray')
    plt.title('gt_mask')

    plt.subplot(1,5,5,adjustable='box')
    plt.imshow(im_show)
    plt.title('overlay')

    plt.show()
    print('mean sad', error_dic['sad'][-1],
          'mean mse', error_dic['mse'][-1],
          'mean psnr', error_dic['psnr'][-1])



#total_error /=len(range(201,292,10))
#print('Mean Error:', np.mean(errors), 'Std: ', np.std(errors))



In [ ]:
print('Mean SAD', np.mean(error_dic['sad']),'+/-', np.std(error_dic['sad']))
print('Mean MSE', np.mean(error_dic['mse']),'+/-', np.std(error_dic['mse']))
print('Mean PSNR', np.mean(error_dic['psnr']),'+/-', np.std(error_dic['psnr']))
plt.subplot(1,3,1),  plt.plot(error_dic['sad']), plt.title('sad')
plt.subplot(1,3,2),  plt.plot(error_dic['mse']), plt.title('mse')
plt.subplot(1,3,3),  plt.plot(error_dic['psnr']), plt.title('psnr')
plt.show()

## 3. Replace the First-Frame Background with a Temporal Median

The original notebook uses `frame_201` as the background reference.

For this first controlled improvement, **all other processing remains unchanged**:

- Gaussian filtering: `sigma=1.0`
- histogram transformation: `saturate(im, 0, 90)` then `saturate(im, 10, 100)`
- signed subtraction: unchanged
- dilation: `disk(2)`
- threshold: `0.1`
- opening: `disk(2)`
- ground-truth construction: unchanged
- original SAD / MSE / PSNR functions: unchanged

Only the reference changes from:

\[
B = I_{201}
\]

to the temporal median:

\[
B_{\mathrm{median}}(x,y)
=
\operatorname{median}_{t} I_t(x,y)
\]



### Build the Temporal-Median Reference

Each frame is first preprocessed exactly as in the original notebook:

```text
frame
  ↓
Gaussian sigma = 1.0
  ↓
saturate(0, 90)
  ↓
processed frame
```

The median is then computed pixel by pixel across the 10 processed frames.


In [ ]:

frame_numbers = list(range(201, 292, 10))

processed_frames = []

for num_f in frame_numbers:
    im_filename = ''.join(
        ['frame_', str(num_f), '.png']
    )
    im_f = os.path.join(IMDIR, im_filename)

    image = io.imread(
        im_f,
        as_gray=True
    )

    # Keep the original preprocessing unchanged.
    image = image.astype(float) / 255.

    im = image.copy()
    im = gaussian(im, sigma=1.0)
    im = saturate(im, 0, 90)

    processed_frames.append(im)

processed_stack = np.stack(
    processed_frames,
    axis=0
)

im_ref_median = np.median(
    processed_stack,
    axis=0
)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(
    processed_frames[0],
    cmap='gray'
)
plt.title('Original reference: frame 201')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(
    im_ref_median,
    cmap='gray'
)
plt.title('Temporal median reference')
plt.axis('off')

plt.tight_layout()
plt.show()



### Run the Same Original Pipeline

Only the background reference is replaced.  
Everything downstream remains identical to the original threshold-based pipeline.


In [ ]:

median_error_dic = {
    'sad': [],
    'mse': [],
    'psnr': []
}

median_results = {}

for num_f in range(201, 292, 10):

    im_filename = ''.join(
        ['frame_', str(num_f), '.png']
    )
    guidewire_filename = ''.join(
        [str(num_f), '_GuideWire.tiff']
    )
    microcath_filename = ''.join(
        [str(num_f), '_MicroCath.tiff']
    )

    im_f = os.path.join(IMDIR, im_filename)
    guidewire_f = os.path.join(
        IMDIR,
        guidewire_filename
    )
    microcath_f = os.path.join(
        IMDIR,
        microcath_filename
    )

    image = io.imread(
        im_f,
        as_gray=True
    )

    guidewire_mask = io.imread(
        guidewire_f
    )
    if len(guidewire_mask.shape) > 2:
        guidewire_mask = guidewire_mask[:, :, 0]

    microcath_mask = io.imread(
        microcath_f
    )
    if len(microcath_mask.shape) > 2:
        microcath_mask = microcath_mask[:, :, 0]

    gt_mask = guidewire_mask.copy()
    gt_mask[
        np.where(microcath_mask < 128)
    ] = 0

    selem = disk(2)
    gt_mask = erosion(gt_mask, selem)

    image = image.astype(float) / 255.
    gt_mask = gt_mask.astype(float) / 255.

    # Original preprocessing.
    im = image.copy()
    im = gaussian(im, sigma=1.0)
    im = saturate(im, 0, 90)

    # ONLY IMPROVEMENT:
    # use temporal median instead of frame 201.
    im = -(im - im_ref_median)
    im_gauss_diff = im.copy()

    # Original morphology.
    selem = disk(2)
    im = dilation(im, selem)

    # Original histogram transformation.
    im = saturate(im, 10, 100)

    # Original threshold.
    im = im > 0.1

    # Original opening.
    selem = disk(2)
    im = opening(im, selem)

    im = 1 - im
    im_mask = im.copy()

    # compute_errors modifies arrays, so pass copies.
    sad_i, mse_i, psnr_i = compute_errors(
        gt_mask.copy(),
        im_mask.copy()
    )

    median_error_dic['sad'].append(sad_i)
    median_error_dic['mse'].append(mse_i)
    median_error_dic['psnr'].append(
        psnr(mse_i)
    )

    median_results[num_f] = {
        'image': image,
        'difference': im_gauss_diff,
        'mask': im_mask,
        'gt_mask': gt_mask,
    }

print(
    'Temporal median — Mean SAD',
    np.mean(median_error_dic['sad']),
    '+/-',
    np.std(median_error_dic['sad'])
)

print(
    'Temporal median — Mean MSE',
    np.mean(median_error_dic['mse']),
    '+/-',
    np.std(median_error_dic['mse'])
)

print(
    'Temporal median — Mean PSNR',
    np.mean(median_error_dic['psnr']),
    '+/-',
    np.std(median_error_dic['psnr'])
)



### Fair Comparison on the Same Frames

The original first-frame method cannot evaluate `frame_201`, because that frame is its own reference.

Therefore the fair comparison uses the same nine frames:

```text
211, 221, 231, 241, 251, 261, 271, 281, 291
```

Frame `201` is reported separately as an additional capability of the temporal-median background.


In [ ]:

baseline_frames = frame_numbers[1:]

median_sad_same = np.array(
    median_error_dic['sad'][1:]
)
median_mse_same = np.array(
    median_error_dic['mse'][1:]
)
median_psnr_same = np.array(
    median_error_dic['psnr'][1:]
)

print('FIRST-FRAME BACKGROUND — 211–291')
print(
    'Mean SAD :',
    np.mean(error_dic['sad'])
)
print(
    'Mean MSE :',
    np.mean(error_dic['mse'])
)
print(
    'Mean PSNR:',
    np.mean(error_dic['psnr'])
)

print()

print('TEMPORAL MEDIAN — SAME 9 FRAMES')
print(
    'Mean SAD :',
    np.mean(median_sad_same)
)
print(
    'Mean MSE :',
    np.mean(median_mse_same)
)
print(
    'Mean PSNR:',
    np.mean(median_psnr_same)
)

print()

print('FRAME 201 — TEMPORAL MEDIAN ONLY')
print(
    'SAD :',
    median_error_dic['sad'][0]
)
print(
    'MSE :',
    median_error_dic['mse'][0]
)
print(
    'PSNR:',
    median_error_dic['psnr'][0]
)


In [ ]:

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(
    baseline_frames,
    error_dic['sad'],
    marker='o',
    label='First frame'
)
plt.plot(
    baseline_frames,
    median_sad_same,
    marker='o',
    label='Temporal median'
)
plt.title('SAD — same 9 frames')
plt.xlabel('Frame')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(
    baseline_frames,
    error_dic['mse'],
    marker='o',
    label='First frame'
)
plt.plot(
    baseline_frames,
    median_mse_same,
    marker='o',
    label='Temporal median'
)
plt.title('MSE — same 9 frames')
plt.xlabel('Frame')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(
    baseline_frames,
    error_dic['psnr'],
    marker='o',
    label='First frame'
)
plt.plot(
    baseline_frames,
    median_psnr_same,
    marker='o',
    label='Temporal median'
)
plt.title('PSNR — same 9 frames')
plt.xlabel('Frame')
plt.legend()

plt.tight_layout()
plt.show()



### Background-Model Ablation Result

This experiment changes **only the background model**.

No conclusion about thresholding, filtering, morphology, or EM should be drawn from this step.

If the temporal median improves the sequence-level results and allows `frame_201` to be processed, it becomes the new background baseline for the next controlled improvement.


## 4. Stabilize the Pre-Subtraction Histogram Transformation

Background-model ablation changed only the background model:

```text
frame_201 reference
        ↓
temporal median reference
```

That improvement is now kept fixed.

For **Radiometric-normalization ablation**, the only new change concerns the histogram transformation performed **before background subtraction**.

The original notebook computes:

```python
saturate(im, 0, 90)
```

independently for every frame.

That means the intensity mapping can vary from frame to frame.

For temporal subtraction, this may introduce artificial differences that are caused by normalization rather than by the moving guidewire or microcatheter.

We therefore test a **fixed contrast mapping** estimated once from the temporal-median background and applied identically to every frame.

Everything else remains unchanged:

- temporal median background from Background-model ablation;
- Gaussian `sigma=1.0`;
- signed subtraction;
- dilation `disk(2)`;
- second `saturate(..., 10, 100)`;
- threshold `0.1`;
- opening `disk(2)`;
- original SAD / MSE / PSNR.



### 1. Fixed Contrast Mapping

Let the temporal-median background be $B$.

We estimate two fixed intensity limits:

\[
L=P_0(B)
\]

and

\[
H=P_{90}(B)
\]

Then every processed frame uses exactly the same transformation:

\[
I'(x,y)
=
\operatorname{clip}
\left(
\frac{I(x,y)-L}
{H-L},
0,
1
\right)
\]

This preserves the original notebook's `0–90` saturation idea while preventing each frame from choosing a different intensity scale.


In [ ]:

# The temporal-median reference from Improvement 1
# was constructed after Gaussian smoothing + per-frame saturation.
#
# For this experiment, we rebuild a median in the Gaussian-smoothed
# domain BEFORE frame-wise saturation, so one fixed mapping can be
# estimated from the background itself.

gaussian_frames = []

for num_f in frame_numbers:
    im_filename = ''.join(
        ['frame_', str(num_f), '.png']
    )

    im_f = os.path.join(
        IMDIR,
        im_filename
    )

    image = io.imread(
        im_f,
        as_gray=True
    )

    image = image.astype(float) / 255.

    smoothed = gaussian(
        image,
        sigma=1.0
    )

    gaussian_frames.append(
        smoothed
    )

gaussian_stack = np.stack(
    gaussian_frames,
    axis=0
)

background_gaussian_median = np.median(
    gaussian_stack,
    axis=0
)

fixed_low = float(
    np.percentile(
        background_gaussian_median,
        0
    )
)

fixed_high = float(
    np.percentile(
        background_gaussian_median,
        90
    )
)

print(
    'Fixed low :',
    fixed_low
)

print(
    'Fixed high:',
    fixed_high
)


In [ ]:

def fixed_saturate_0_90(
    image,
    low=fixed_low,
    high=fixed_high
):
    image = np.asarray(
        image,
        dtype=float
    )

    return np.clip(
        (
            image - low
        )
        / (
            high - low + 1e-12
        ),
        0.0,
        1.0
    )


fixed_background = fixed_saturate_0_90(
    background_gaussian_median
)

plt.figure(
    figsize=(10, 4)
)

plt.subplot(
    1,
    2,
    1
)

plt.imshow(
    im_ref_median,
    cmap='gray'
)

plt.title(
    'Step 1 median reference'
)

plt.axis(
    'off'
)

plt.subplot(
    1,
    2,
    2
)

plt.imshow(
    fixed_background,
    cmap='gray'
)

plt.title(
    'Step 2 fixed-mapping background'
)

plt.axis(
    'off'
)

plt.tight_layout()
plt.show()



### 2. Same Pipeline with Fixed Pre-Subtraction Mapping

The pipeline remains identical to Background-model ablation except for one controlled replacement:

### Background-model ablation

```python
im = saturate(im, 0, 90)
```

computed independently for each frame.

### Radiometric-normalization ablation

```python
im = fixed_saturate_0_90(im)
```

using the same `fixed_low` and `fixed_high` for every frame.

The second saturation performed after dilation remains exactly as in the original notebook.


In [ ]:

fixed_hist_error_dic = {
    'sad': [],
    'mse': [],
    'psnr': []
}

fixed_hist_results = {}

for num_f in range(201, 292, 10):

    im_filename = ''.join(
        ['frame_', str(num_f), '.png']
    )

    guidewire_filename = ''.join(
        [str(num_f), '_GuideWire.tiff']
    )

    microcath_filename = ''.join(
        [str(num_f), '_MicroCath.tiff']
    )

    im_f = os.path.join(
        IMDIR,
        im_filename
    )

    guidewire_f = os.path.join(
        IMDIR,
        guidewire_filename
    )

    microcath_f = os.path.join(
        IMDIR,
        microcath_filename
    )

    image = io.imread(
        im_f,
        as_gray=True
    )

    guidewire_mask = io.imread(
        guidewire_f
    )

    if len(
        guidewire_mask.shape
    ) > 2:
        guidewire_mask = guidewire_mask[:, :, 0]

    microcath_mask = io.imread(
        microcath_f
    )

    if len(
        microcath_mask.shape
    ) > 2:
        microcath_mask = microcath_mask[:, :, 0]

    gt_mask = guidewire_mask.copy()

    gt_mask[
        np.where(
            microcath_mask < 128
        )
    ] = 0

    # Keep original ground-truth thickening unchanged.
    selem = disk(2)

    gt_mask = erosion(
        gt_mask,
        selem
    )

    image = image.astype(float) / 255.
    gt_mask = gt_mask.astype(float) / 255.

    # Keep original Gaussian filtering unchanged.
    im = gaussian(
        image.copy(),
        sigma=1.0
    )

    # ONLY NEW CHANGE:
    # fixed pre-subtraction histogram mapping.
    im = fixed_saturate_0_90(
        im
    )

    # Keep Improvement 1 background model.
    im = -(
        im
        - fixed_background
    )

    im_gauss_diff = im.copy()

    # Keep original dilation.
    selem = disk(2)

    im = dilation(
        im,
        selem
    )

    # Keep original second histogram transformation.
    im = saturate(
        im,
        10,
        100
    )

    # Keep original threshold.
    im = (
        im > 0.1
    )

    # Keep original opening.
    selem = disk(2)

    im = opening(
        im,
        selem
    )

    im = 1 - im
    im_mask = im.copy()

    sad_i, mse_i, psnr_i = (
        compute_errors(
            gt_mask.copy(),
            im_mask.copy()
        )
    )

    fixed_hist_error_dic[
        'sad'
    ].append(
        sad_i
    )

    fixed_hist_error_dic[
        'mse'
    ].append(
        mse_i
    )

    fixed_hist_error_dic[
        'psnr'
    ].append(
        psnr(
            mse_i
        )
    )

    fixed_hist_results[
        num_f
    ] = {
        'image':
            image,
        'difference':
            im_gauss_diff,
        'mask':
            im_mask,
        'gt_mask':
            gt_mask,
    }

print(
    'Step 2 — Mean SAD ',
    np.mean(
        fixed_hist_error_dic[
            'sad'
        ]
    )
)

print(
    'Step 2 — Mean MSE ',
    np.mean(
        fixed_hist_error_dic[
            'mse'
        ]
    )
)

print(
    'Step 2 — Mean PSNR',
    np.mean(
        fixed_hist_error_dic[
            'psnr'
        ]
    )
)



### 3. Background-model ablation vs Radiometric-normalization ablation

This is the key controlled comparison.

Both methods use:

- the temporal background;
- the same Gaussian filter;
- the same subtraction direction;
- the same dilation;
- the same second saturation;
- the same threshold;
- the same opening.

Only the **pre-subtraction intensity mapping** differs.


In [ ]:

step1_sad = np.array(
    median_error_dic[
        'sad'
    ]
)

step1_mse = np.array(
    median_error_dic[
        'mse'
    ]
)

step1_psnr = np.array(
    median_error_dic[
        'psnr'
    ]
)

step2_sad = np.array(
    fixed_hist_error_dic[
        'sad'
    ]
)

step2_mse = np.array(
    fixed_hist_error_dic[
        'mse'
    ]
)

step2_psnr = np.array(
    fixed_hist_error_dic[
        'psnr'
    ]
)

print(
    'IMPROVEMENT 1 — TEMPORAL MEDIAN '
    '+ ORIGINAL PER-FRAME SATURATION'
)

print(
    'Mean SAD :',
    np.mean(
        step1_sad
    )
)

print(
    'Mean MSE :',
    np.mean(
        step1_mse
    )
)

print(
    'Mean PSNR:',
    np.mean(
        step1_psnr
    )
)

print()

print(
    'IMPROVEMENT 2 — TEMPORAL MEDIAN '
    '+ FIXED SATURATION'
)

print(
    'Mean SAD :',
    np.mean(
        step2_sad
    )
)

print(
    'Mean MSE :',
    np.mean(
        step2_mse
    )
)

print(
    'Mean PSNR:',
    np.mean(
        step2_psnr
    )
)


In [ ]:

plt.figure(
    figsize=(12, 4)
)

plt.subplot(
    1,
    3,
    1
)

plt.plot(
    frame_numbers,
    step1_sad,
    marker='o',
    label='Step 1'
)

plt.plot(
    frame_numbers,
    step2_sad,
    marker='o',
    label='Step 2'
)

plt.title(
    'SAD'
)

plt.xlabel(
    'Frame'
)

plt.legend()

plt.subplot(
    1,
    3,
    2
)

plt.plot(
    frame_numbers,
    step1_mse,
    marker='o',
    label='Step 1'
)

plt.plot(
    frame_numbers,
    step2_mse,
    marker='o',
    label='Step 2'
)

plt.title(
    'MSE'
)

plt.xlabel(
    'Frame'
)

plt.legend()

plt.subplot(
    1,
    3,
    3
)

plt.plot(
    frame_numbers,
    step1_psnr,
    marker='o',
    label='Step 1'
)

plt.plot(
    frame_numbers,
    step2_psnr,
    marker='o',
    label='Step 2'
)

plt.title(
    'PSNR'
)

plt.xlabel(
    'Frame'
)

plt.legend()

plt.tight_layout()
plt.show()



### Radiometric-Mapping Retention Criterion

We do **not** keep the fixed histogram transformation merely because it is theoretically cleaner.

We retain it only if the sequence-level results support it.

Decision criteria:

- lower SAD is better;
- lower MSE is better;
- higher PSNR is better;
- visual mask quality must remain coherent.

If Step 2 is worse, the scientifically correct decision is to keep the Step-1 transformation and move to the next component.

Retention is based on controlled-ablation evidence rather than algorithmic complexity or visual preference.


## 5. Optimize Spatial Gaussian Filtering

The retained pipeline already contains:

1. temporal-median background;
2. fixed pre-subtraction histogram mapping.

For **Spatial-filtering ablation**, only the Gaussian spatial-filter scale changes.

Everything else remains fixed:

- temporal-median background;
- fixed histogram mapping;
- signed subtraction;
- dilation `disk(2)`;
- second saturation `saturate(..., 10, 100)`;
- threshold `0.1`;
- opening `disk(2)`;
- original SAD / MSE / PSNR.

The original notebook uses `sigma=1.0`.

We test:

\[
\sigma \in \{0.5,\;0.75,\;1.0,\;1.25,\;1.5,\;2.0\}
\]


In [ ]:

SIGMA_CANDIDATES = [
    0.5,
    0.75,
    1.0,
    1.25,
    1.5,
    2.0,
]

print("Sigma candidates:", SIGMA_CANDIDATES)



### Controlled Sigma Evaluation

For every candidate sigma, the background is rebuilt in the **same Gaussian domain** as the current frame.  
This avoids comparing a frame filtered with one sigma against a background filtered with another.

Only sigma changes; all later operations remain identical.


In [ ]:

def evaluate_sigma(sigma_value):

    error_dic_sigma = {
        'sad': [],
        'mse': [],
        'psnr': []
    }

    # Build temporal-median background in this sigma domain.
    smoothed_frames = []

    for num_f in frame_numbers:
        im_filename = ''.join(
            ['frame_', str(num_f), '.png']
        )
        im_f = os.path.join(IMDIR, im_filename)

        image = io.imread(
            im_f,
            as_gray=True
        )

        image = image.astype(float) / 255.

        smoothed = gaussian(
            image,
            sigma=sigma_value
        )

        smoothed_frames.append(smoothed)

    smoothed_stack = np.stack(
        smoothed_frames,
        axis=0
    )

    background_sigma = np.median(
        smoothed_stack,
        axis=0
    )

    # Same Step-2 idea:
    # one fixed mapping derived from the background.
    low_sigma = float(
        np.percentile(
            background_sigma,
            0
        )
    )

    high_sigma = float(
        np.percentile(
            background_sigma,
            90
        )
    )

    def fixed_map_sigma(image):
        return np.clip(
            (image - low_sigma)
            / (
                high_sigma
                - low_sigma
                + 1e-12
            ),
            0.0,
            1.0
        )

    background_sigma = fixed_map_sigma(
        background_sigma
    )

    for num_f in frame_numbers:

        im_filename = ''.join(
            ['frame_', str(num_f), '.png']
        )
        guidewire_filename = ''.join(
            [str(num_f), '_GuideWire.tiff']
        )
        microcath_filename = ''.join(
            [str(num_f), '_MicroCath.tiff']
        )

        im_f = os.path.join(
            IMDIR,
            im_filename
        )
        guidewire_f = os.path.join(
            IMDIR,
            guidewire_filename
        )
        microcath_f = os.path.join(
            IMDIR,
            microcath_filename
        )

        image = io.imread(
            im_f,
            as_gray=True
        )

        guidewire_mask = io.imread(
            guidewire_f
        )
        if len(guidewire_mask.shape) > 2:
            guidewire_mask = guidewire_mask[:, :, 0]

        microcath_mask = io.imread(
            microcath_f
        )
        if len(microcath_mask.shape) > 2:
            microcath_mask = microcath_mask[:, :, 0]

        gt_mask = guidewire_mask.copy()

        gt_mask[
            np.where(
                microcath_mask < 128
            )
        ] = 0

        # Keep original GT thickening.
        selem = disk(2)
        gt_mask = erosion(
            gt_mask,
            selem
        )

        image = image.astype(float) / 255.
        gt_mask = gt_mask.astype(float) / 255.

        # ONLY controlled variable:
        # Gaussian sigma.
        im = gaussian(
            image.copy(),
            sigma=sigma_value
        )

        # Step 2 retained.
        im = fixed_map_sigma(im)

        # Step 1 retained.
        im = -(im - background_sigma)

        # Original downstream processing.
        selem = disk(2)
        im = dilation(im, selem)

        im = saturate(
            im,
            10,
            100
        )

        im = im > 0.1

        selem = disk(2)
        im = opening(im, selem)

        im = 1 - im
        im_mask = im.copy()

        sad_i, mse_i, psnr_i = compute_errors(
            gt_mask.copy(),
            im_mask.copy()
        )

        error_dic_sigma['sad'].append(
            sad_i
        )
        error_dic_sigma['mse'].append(
            mse_i
        )
        error_dic_sigma['psnr'].append(
            psnr(mse_i)
        )

    return error_dic_sigma


In [ ]:

sigma_results = {}

for sigma_value in SIGMA_CANDIDATES:
    sigma_results[sigma_value] = evaluate_sigma(
        sigma_value
    )

print(
    f"{'Sigma':>7s} "
    f"{'Mean SAD':>12s} "
    f"{'Mean MSE':>12s} "
    f"{'Mean PSNR':>12s}"
)

print("-" * 48)

for sigma_value in SIGMA_CANDIDATES:
    result = sigma_results[sigma_value]

    print(
        f"{sigma_value:7.2f} "
        f"{np.mean(result['sad']):12.6f} "
        f"{np.mean(result['mse']):12.6f} "
        f"{np.mean(result['psnr']):12.6f}"
    )



### Select the Best Sigma

We select the candidate with the lowest mean MSE.  
Because the masks are binary, SAD and MSE should lead to the same ranking.


In [ ]:

mean_mse_by_sigma = {
    sigma_value: float(
        np.mean(
            sigma_results[
                sigma_value
            ]['mse']
        )
    )
    for sigma_value in SIGMA_CANDIDATES
}

BEST_SIGMA = min(
    mean_mse_by_sigma,
    key=mean_mse_by_sigma.get
)

best_sigma_result = sigma_results[
    BEST_SIGMA
]

print("Best sigma:", BEST_SIGMA)
print(
    "Mean SAD :",
    np.mean(
        best_sigma_result['sad']
    )
)
print(
    "Mean MSE :",
    np.mean(
        best_sigma_result['mse']
    )
)
print(
    "Mean PSNR:",
    np.mean(
        best_sigma_result['psnr']
    )
)


In [ ]:

mean_sad_values = [
    np.mean(
        sigma_results[s]['sad']
    )
    for s in SIGMA_CANDIDATES
]

mean_mse_values = [
    np.mean(
        sigma_results[s]['mse']
    )
    for s in SIGMA_CANDIDATES
]

mean_psnr_values = [
    np.mean(
        sigma_results[s]['psnr']
    )
    for s in SIGMA_CANDIDATES
]

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(
    SIGMA_CANDIDATES,
    mean_sad_values,
    marker='o'
)
plt.axvline(
    BEST_SIGMA,
    linestyle='--'
)
plt.title('Mean SAD vs sigma')
plt.xlabel('Gaussian sigma')

plt.subplot(1, 3, 2)
plt.plot(
    SIGMA_CANDIDATES,
    mean_mse_values,
    marker='o'
)
plt.axvline(
    BEST_SIGMA,
    linestyle='--'
)
plt.title('Mean MSE vs sigma')
plt.xlabel('Gaussian sigma')

plt.subplot(1, 3, 3)
plt.plot(
    SIGMA_CANDIDATES,
    mean_psnr_values,
    marker='o'
)
plt.axvline(
    BEST_SIGMA,
    linestyle='--'
)
plt.title('Mean PSNR vs sigma')
plt.xlabel('Gaussian sigma')

plt.tight_layout()
plt.show()



### Step 2 vs Step 3

Step 2 uses the original `sigma=1.0`.

Step 3 keeps every other retained component unchanged and compares that value against the best candidate found above.


In [ ]:

print('STEP 2 — sigma = 1.0')
print(
    'Mean SAD :',
    np.mean(
        fixed_hist_error_dic['sad']
    )
)
print(
    'Mean MSE :',
    np.mean(
        fixed_hist_error_dic['mse']
    )
)
print(
    'Mean PSNR:',
    np.mean(
        fixed_hist_error_dic['psnr']
    )
)

print()

print(
    f'STEP 3 — sigma = {BEST_SIGMA}'
)
print(
    'Mean SAD :',
    np.mean(
        best_sigma_result['sad']
    )
)
print(
    'Mean MSE :',
    np.mean(
        best_sigma_result['mse']
    )
)
print(
    'Mean PSNR:',
    np.mean(
        best_sigma_result['psnr']
    )
)



### Step-3 Decision Rule

If a new sigma improves the sequence-level metrics, it becomes the retained value.

If `sigma=1.0` remains best, then Spatial-filtering ablation still succeeds scientifically:

> the original Gaussian scale was already appropriate and is now quantitatively justified.


## 6. Add and Tune Spectral-Domain High-Pass Filtering

The retained pipeline now contains:

1. **Background-model ablation** — temporal-median background;
2. **Radiometric-normalization ablation** — fixed pre-subtraction histogram mapping;
3. **Spatial-filtering ablation** — Gaussian spatial filtering with `sigma = 0.75`.

For **Spectral-filtering ablation**, only one new processing family is introduced:

> **frequency-domain Gaussian high-pass filtering**

Everything else remains fixed:

- temporal-median background;
- fixed histogram mapping;
- Gaussian `sigma = 0.75`;
- signed subtraction;
- dilation `disk(2)`;
- second saturation `saturate(..., 10, 100)`;
- threshold `0.1`;
- opening `disk(2)`;
- original SAD / MSE / PSNR.



### 1. Why Add a Spectral Filter?

After background subtraction, slowly varying residual structures are concentrated near low spatial frequencies.

The guidewire and microcatheter are thin structures and therefore contain stronger high-frequency content.

A spectral high-pass filter can suppress slow residual variation while retaining thin structures.

We use a Gaussian high-pass filter:

\[
H(u,v)
=
1-
\exp
\left(
-\frac{D(u,v)^2}
{2D_0^2}
\right)
\]

where:

- \(D(u,v)\) is the distance to the center of the frequency plane;
- \(D_0\) is the cutoff parameter.

Small \(D_0\):

- removes only a narrow low-frequency region.

Large \(D_0\):

- suppresses a broader frequency range;
- can remove useful tool information.

We therefore measure the effect instead of choosing \(D_0\) arbitrarily.


In [ ]:

SPECTRAL_CUTOFFS = [
    5.0,
    7.5,
    10.0,
    12.5,
    20.0,
    40.0,
    80.0,
]

print(
    "Spectral cutoff candidates:",
    SPECTRAL_CUTOFFS
)



### 2. Gaussian High-Pass Filter

The Fourier-processing sequence is:

```text
residual image
    ↓
2-D FFT
    ↓
fftshift
    ↓
multiply by Gaussian HPF
    ↓
ifftshift
    ↓
inverse 2-D FFT
    ↓
real high-frequency response
```

Only the positive spectral response is retained before the original morphology and segmentation stages.

This preserves the original signed interpretation:

> positive response = candidate darker moving structure.


In [ ]:

def gaussian_high_pass(
    shape,
    cutoff
):
    rows, cols = shape

    cy = rows // 2
    cx = cols // 2

    y, x = np.ogrid[
        :rows,
        :cols
    ]

    distance_squared = (
        (y - cy) ** 2
        + (x - cx) ** 2
    )

    low_pass = np.exp(
        -distance_squared
        / (
            2.0
            * cutoff ** 2
        )
    )

    return 1.0 - low_pass


def apply_spectral_high_pass(
    residual,
    cutoff
):
    spectrum = np.fft.fftshift(
        np.fft.fft2(
            residual
        )
    )

    hpf = gaussian_high_pass(
        residual.shape,
        cutoff
    )

    filtered_spectrum = (
        spectrum
        * hpf
    )

    filtered = np.real(
        np.fft.ifft2(
            np.fft.ifftshift(
                filtered_spectrum
            )
        )
    )

    # Preserve only positive evidence
    # for darker moving structures.
    filtered = np.maximum(
        filtered,
        0.0
    )

    return (
        filtered,
        hpf,
        spectrum
    )



### 3. Visualize the Spectrum and Filter

Before evaluating the entire sequence, inspect one representative frame.

This is only a visualization step; parameter selection is still based on sequence-level metrics.


In [ ]:

REPRESENTATIVE_FRAME = 251

rep_index = frame_numbers.index(
    REPRESENTATIVE_FRAME
)

rep_image = io.imread(
    os.path.join(
        IMDIR,
        f"frame_{REPRESENTATIVE_FRAME}.png"
    ),
    as_gray=True
)

rep_image = (
    rep_image.astype(float)
    / 255.
)

rep_smoothed = gaussian(
    rep_image,
    sigma=BEST_SIGMA
)

# Rebuild Step-3 background for the retained sigma.
step3_smoothed_frames = []

for num_f in frame_numbers:
    image = io.imread(
        os.path.join(
            IMDIR,
            f"frame_{num_f}.png"
        ),
        as_gray=True
    )

    image = (
        image.astype(float)
        / 255.
    )

    step3_smoothed_frames.append(
        gaussian(
            image,
            sigma=BEST_SIGMA
        )
    )

step3_background_raw = np.median(
    np.stack(
        step3_smoothed_frames,
        axis=0
    ),
    axis=0
)

step3_low = float(
    np.percentile(
        step3_background_raw,
        0
    )
)

step3_high = float(
    np.percentile(
        step3_background_raw,
        90
    )
)


def step3_fixed_map(
    image
):
    return np.clip(
        (
            image - step3_low
        )
        / (
            step3_high
            - step3_low
            + 1e-12
        ),
        0.0,
        1.0
    )


step3_background = step3_fixed_map(
    step3_background_raw
)

rep_mapped = step3_fixed_map(
    rep_smoothed
)

rep_residual = -(
    rep_mapped
    - step3_background
)

(
    rep_spectral,
    rep_hpf,
    rep_spectrum
) = apply_spectral_high_pass(
    rep_residual,
    cutoff=10.0
)

plt.figure(
    figsize=(12, 8)
)

plt.subplot(
    2,
    2,
    1
)

plt.imshow(
    rep_residual,
    cmap='gray'
)

plt.title(
    'Step-3 residual'
)

plt.axis(
    'off'
)

plt.subplot(
    2,
    2,
    2
)

plt.imshow(
    np.log1p(
        np.abs(
            rep_spectrum
        )
    ),
    cmap='gray'
)

plt.title(
    'Log magnitude spectrum'
)

plt.axis(
    'off'
)

plt.subplot(
    2,
    2,
    3
)

plt.imshow(
    rep_hpf,
    cmap='gray'
)

plt.title(
    'Gaussian HPF — D0=10'
)

plt.axis(
    'off'
)

plt.subplot(
    2,
    2,
    4
)

plt.imshow(
    rep_spectral,
    cmap='gray'
)

plt.title(
    'Spectrally filtered residual'
)

plt.axis(
    'off'
)

plt.tight_layout()
plt.show()



### 4. Controlled Cutoff Evaluation

For each candidate \(D_0\):

```text
input frame
    ↓
Gaussian spatial filter sigma = 0.75
    ↓
fixed histogram mapping
    ↓
temporal-median subtraction
    ↓
Gaussian spectral high-pass
    ↓
original dilation
    ↓
original second saturation
    ↓
original threshold 0.1
    ↓
original opening
    ↓
SAD / MSE / PSNR
```

Only the spectral cutoff changes.


In [ ]:

def evaluate_spectral_cutoff(
    cutoff
):
    error_dic_cutoff = {
        'sad': [],
        'mse': [],
        'psnr': []
    }

    for num_f in frame_numbers:

        im_filename = ''.join(
            [
                'frame_',
                str(num_f),
                '.png'
            ]
        )

        guidewire_filename = ''.join(
            [
                str(num_f),
                '_GuideWire.tiff'
            ]
        )

        microcath_filename = ''.join(
            [
                str(num_f),
                '_MicroCath.tiff'
            ]
        )

        im_f = os.path.join(
            IMDIR,
            im_filename
        )

        guidewire_f = os.path.join(
            IMDIR,
            guidewire_filename
        )

        microcath_f = os.path.join(
            IMDIR,
            microcath_filename
        )

        image = io.imread(
            im_f,
            as_gray=True
        )

        guidewire_mask = io.imread(
            guidewire_f
        )

        if len(
            guidewire_mask.shape
        ) > 2:
            guidewire_mask = (
                guidewire_mask[:, :, 0]
            )

        microcath_mask = io.imread(
            microcath_f
        )

        if len(
            microcath_mask.shape
        ) > 2:
            microcath_mask = (
                microcath_mask[:, :, 0]
            )

        gt_mask = (
            guidewire_mask.copy()
        )

        gt_mask[
            np.where(
                microcath_mask < 128
            )
        ] = 0

        # Original ground-truth thickening.
        selem = disk(2)

        gt_mask = erosion(
            gt_mask,
            selem
        )

        image = (
            image.astype(float)
            / 255.
        )

        gt_mask = (
            gt_mask.astype(float)
            / 255.
        )

        # Improvement 3 retained.
        im = gaussian(
            image.copy(),
            sigma=BEST_SIGMA
        )

        # Improvement 2 retained.
        im = step3_fixed_map(
            im
        )

        # Improvement 1 retained.
        im = -(
            im
            - step3_background
        )

        # ONLY NEW COMPONENT:
        # spectral Gaussian high-pass.
        im, _, _ = (
            apply_spectral_high_pass(
                im,
                cutoff=cutoff
            )
        )

        # Original downstream processing.
        selem = disk(2)

        im = dilation(
            im,
            selem
        )

        im = saturate(
            im,
            10,
            100
        )

        im = (
            im > 0.1
        )

        selem = disk(2)

        im = opening(
            im,
            selem
        )

        im = 1 - im
        im_mask = im.copy()

        sad_i, mse_i, psnr_i = (
            compute_errors(
                gt_mask.copy(),
                im_mask.copy()
            )
        )

        error_dic_cutoff[
            'sad'
        ].append(
            sad_i
        )

        error_dic_cutoff[
            'mse'
        ].append(
            mse_i
        )

        error_dic_cutoff[
            'psnr'
        ].append(
            psnr(
                mse_i
            )
        )

    return error_dic_cutoff


In [ ]:

spectral_results = {}

for cutoff in SPECTRAL_CUTOFFS:
    spectral_results[
        cutoff
    ] = evaluate_spectral_cutoff(
        cutoff
    )

print(
    f"{'D0':>8s} "
    f"{'Mean SAD':>12s} "
    f"{'Mean MSE':>12s} "
    f"{'Mean PSNR':>12s}"
)

print(
    "-" * 49
)

for cutoff in SPECTRAL_CUTOFFS:
    result = spectral_results[
        cutoff
    ]

    print(
        f"{cutoff:8.2f} "
        f"{np.mean(result['sad']):12.6f} "
        f"{np.mean(result['mse']):12.6f} "
        f"{np.mean(result['psnr']):12.6f}"
    )



### 5. Select the Best Spectral Cutoff

The same selection rule is preserved:

- lower mean SAD;
- lower mean MSE;
- higher mean PSNR.

The retained \(D_0\) is selected by minimum mean MSE.


In [ ]:

mean_mse_by_cutoff = {
    cutoff:
        float(
            np.mean(
                spectral_results[
                    cutoff
                ][
                    'mse'
                ]
            )
        )
    for cutoff in SPECTRAL_CUTOFFS
}

BEST_SPECTRAL_CUTOFF = min(
    mean_mse_by_cutoff,
    key=mean_mse_by_cutoff.get
)

best_spectral_result = (
    spectral_results[
        BEST_SPECTRAL_CUTOFF
    ]
)

print(
    "Best spectral cutoff:",
    BEST_SPECTRAL_CUTOFF
)

print(
    "Mean SAD :",
    np.mean(
        best_spectral_result[
            'sad'
        ]
    )
)

print(
    "Mean MSE :",
    np.mean(
        best_spectral_result[
            'mse'
        ]
    )
)

print(
    "Mean PSNR:",
    np.mean(
        best_spectral_result[
            'psnr'
        ]
    )
)


In [ ]:

spectral_mean_sad = [
    np.mean(
        spectral_results[
            cutoff
        ][
            'sad'
        ]
    )
    for cutoff in SPECTRAL_CUTOFFS
]

spectral_mean_mse = [
    np.mean(
        spectral_results[
            cutoff
        ][
            'mse'
        ]
    )
    for cutoff in SPECTRAL_CUTOFFS
]

spectral_mean_psnr = [
    np.mean(
        spectral_results[
            cutoff
        ][
            'psnr'
        ]
    )
    for cutoff in SPECTRAL_CUTOFFS
]

plt.figure(
    figsize=(12, 4)
)

plt.subplot(
    1,
    3,
    1
)

plt.plot(
    SPECTRAL_CUTOFFS,
    spectral_mean_sad,
    marker='o'
)

plt.axvline(
    BEST_SPECTRAL_CUTOFF,
    linestyle='--'
)

plt.title(
    'Mean SAD vs D0'
)

plt.xlabel(
    'Gaussian HPF cutoff'
)

plt.subplot(
    1,
    3,
    2
)

plt.plot(
    SPECTRAL_CUTOFFS,
    spectral_mean_mse,
    marker='o'
)

plt.axvline(
    BEST_SPECTRAL_CUTOFF,
    linestyle='--'
)

plt.title(
    'Mean MSE vs D0'
)

plt.xlabel(
    'Gaussian HPF cutoff'
)

plt.subplot(
    1,
    3,
    3
)

plt.plot(
    SPECTRAL_CUTOFFS,
    spectral_mean_psnr,
    marker='o'
)

plt.axvline(
    BEST_SPECTRAL_CUTOFF,
    linestyle='--'
)

plt.title(
    'Mean PSNR vs D0'
)

plt.xlabel(
    'Gaussian HPF cutoff'
)

plt.tight_layout()
plt.show()



### Spatial-Only vs Spectral-Augmented Comparison

This comparison determines whether the spectral stage should actually be retained.

Step 3:

```text
temporal median
+ fixed histogram mapping
+ Gaussian sigma = 0.75
+ no spectral filtering
```

Step 4:

```text
same pipeline
+ Gaussian spectral high-pass
```


In [ ]:

print(
    'STEP 3 — NO SPECTRAL FILTER'
)

print(
    'Mean SAD :',
    np.mean(
        best_sigma_result[
            'sad'
        ]
    )
)

print(
    'Mean MSE :',
    np.mean(
        best_sigma_result[
            'mse'
        ]
    )
)

print(
    'Mean PSNR:',
    np.mean(
        best_sigma_result[
            'psnr'
        ]
    )
)

print()

print(
    'STEP 4 — GAUSSIAN HPF '
    f'D0 = {BEST_SPECTRAL_CUTOFF}'
)

print(
    'Mean SAD :',
    np.mean(
        best_spectral_result[
            'sad'
        ]
    )
)

print(
    'Mean MSE :',
    np.mean(
        best_spectral_result[
            'mse'
        ]
    )
)

print(
    'Mean PSNR:',
    np.mean(
        best_spectral_result[
            'psnr'
        ]
    )
)



### Step-4 Decision Rule

The spectral filter is retained only if it improves the measured sequence-level performance.

A high-pass cutoff that is too large removes useful low-to-mid-frequency information and can severely degrade the segmentation.

Therefore the experiment must justify both:

1. whether spectral filtering should be used;
2. which cutoff is appropriate.

If the selected spectral filter improves the metrics, it becomes part of the retained pipeline for the next controlled improvement.


## 7. Optimize Morphological Refinement

The retained pipeline is now fixed at:

- temporal-median background;
- fixed pre-subtraction histogram mapping;
- Gaussian spatial filter `sigma = 0.75`;
- Gaussian high-pass spectral filter `D0 = 10`.

For this step, only morphology changes.

The original notebook uses:

```python
dilation(..., disk(2))
opening(..., disk(2))
```

We test dilation and opening radii from 0 to 4. A radius of 0 means that the operation is skipped.


In [ ]:

DILATION_RADII = [0, 1, 2, 3, 4]
OPENING_RADII = [0, 1, 2, 3, 4]

print("Dilation radii:", DILATION_RADII)
print("Opening radii :", OPENING_RADII)



### Reusable Morphology Evaluation

Only `(dilation_radius, opening_radius)` changes. Every earlier processing stage and all final thresholding parameters remain unchanged.


In [ ]:

def evaluate_morphology(dilation_radius, opening_radius):
    errors = {
        'sad': [],
        'mse': [],
        'psnr': []
    }

    for num_f in frame_numbers:
        im_f = os.path.join(
            IMDIR,
            f"frame_{num_f}.png"
        )
        guidewire_f = os.path.join(
            IMDIR,
            f"{num_f}_GuideWire.tiff"
        )
        microcath_f = os.path.join(
            IMDIR,
            f"{num_f}_MicroCath.tiff"
        )

        image = io.imread(
            im_f,
            as_gray=True
        )

        guidewire_mask = io.imread(
            guidewire_f
        )
        if len(guidewire_mask.shape) > 2:
            guidewire_mask = guidewire_mask[:, :, 0]

        microcath_mask = io.imread(
            microcath_f
        )
        if len(microcath_mask.shape) > 2:
            microcath_mask = microcath_mask[:, :, 0]

        gt_mask = guidewire_mask.copy()
        gt_mask[
            np.where(
                microcath_mask < 128
            )
        ] = 0

        # Ground-truth thickening remains unchanged.
        gt_mask = erosion(
            gt_mask,
            disk(2)
        )

        image = image.astype(float) / 255.
        gt_mask = gt_mask.astype(float) / 255.

        # Step 3 retained.
        im = gaussian(
            image.copy(),
            sigma=BEST_SIGMA
        )

        # Step 2 retained.
        im = step3_fixed_map(im)

        # Step 1 retained.
        im = -(im - step3_background)

        # Step 4 retained.
        im, _, _ = apply_spectral_high_pass(
            im,
            cutoff=BEST_SPECTRAL_CUTOFF
        )

        # Only controlled morphology parameter 1.
        if dilation_radius > 0:
            im = dilation(
                im,
                disk(dilation_radius)
            )

        # Original downstream contrast transform.
        im = saturate(
            im,
            10,
            100
        )

        # Original threshold.
        im = im > 0.1

        # Only controlled morphology parameter 2.
        if opening_radius > 0:
            im = opening(
                im,
                disk(opening_radius)
            )

        im = 1 - im
        im_mask = im.copy()

        sad_i, mse_i, _ = compute_errors(
            gt_mask.copy(),
            im_mask.copy()
        )

        errors['sad'].append(sad_i)
        errors['mse'].append(mse_i)
        errors['psnr'].append(
            psnr(mse_i)
        )

    return errors



### 1. Dilation Sweep

Opening remains at its original radius `2`. Only dilation changes.


In [ ]:

dilation_results = {}

for radius in DILATION_RADII:
    dilation_results[radius] = evaluate_morphology(
        dilation_radius=radius,
        opening_radius=2
    )

print(
    f"{'Dil. r':>8s} "
    f"{'Mean SAD':>12s} "
    f"{'Mean MSE':>12s} "
    f"{'Mean PSNR':>12s}"
)
print("-" * 49)

for radius in DILATION_RADII:
    result = dilation_results[radius]
    print(
        f"{radius:8d} "
        f"{np.mean(result['sad']):12.6f} "
        f"{np.mean(result['mse']):12.6f} "
        f"{np.mean(result['psnr']):12.6f}"
    )

BEST_DILATION_RADIUS = min(
    DILATION_RADII,
    key=lambda r: np.mean(
        dilation_results[r]['mse']
    )
)

print(
    "\nBest dilation radius:",
    BEST_DILATION_RADIUS
)



### 2. Opening Sweep

The best dilation radius is now fixed. Only opening changes.


In [ ]:

opening_results = {}

for radius in OPENING_RADII:
    opening_results[radius] = evaluate_morphology(
        dilation_radius=BEST_DILATION_RADIUS,
        opening_radius=radius
    )

print(
    f"{'Open r':>8s} "
    f"{'Mean SAD':>12s} "
    f"{'Mean MSE':>12s} "
    f"{'Mean PSNR':>12s}"
)
print("-" * 49)

for radius in OPENING_RADII:
    result = opening_results[radius]
    print(
        f"{radius:8d} "
        f"{np.mean(result['sad']):12.6f} "
        f"{np.mean(result['mse']):12.6f} "
        f"{np.mean(result['psnr']):12.6f}"
    )

BEST_OPENING_RADIUS = min(
    OPENING_RADII,
    key=lambda r: np.mean(
        opening_results[r]['mse']
    )
)

print(
    "\nBest opening radius:",
    BEST_OPENING_RADIUS
)



### 3. Full Grid Verification

Because dilation and opening interact, a complete 5×5 grid verifies that the sequential selection did not miss a better pair.


In [ ]:

morphology_grid = {}

for d_radius in DILATION_RADII:
    for o_radius in OPENING_RADII:
        morphology_grid[
            (d_radius, o_radius)
        ] = evaluate_morphology(
            dilation_radius=d_radius,
            opening_radius=o_radius
        )

BEST_MORPHOLOGY_PAIR = min(
    morphology_grid,
    key=lambda pair: np.mean(
        morphology_grid[pair]['mse']
    )
)

best_morphology_result = morphology_grid[
    BEST_MORPHOLOGY_PAIR
]

print(
    "Best pair (dilation, opening):",
    BEST_MORPHOLOGY_PAIR
)

print(
    "Mean SAD :",
    np.mean(
        best_morphology_result['sad']
    )
)
print(
    "Mean MSE :",
    np.mean(
        best_morphology_result['mse']
    )
)
print(
    "Mean PSNR:",
    np.mean(
        best_morphology_result['psnr']
    )
)


In [ ]:

mse_grid = np.zeros(
    (
        len(DILATION_RADII),
        len(OPENING_RADII)
    )
)

for i, d_radius in enumerate(DILATION_RADII):
    for j, o_radius in enumerate(OPENING_RADII):
        mse_grid[i, j] = np.mean(
            morphology_grid[
                (d_radius, o_radius)
            ]['mse']
        )

plt.figure(figsize=(7, 6))
plt.imshow(
    mse_grid,
    cmap='viridis',
    aspect='auto'
)
plt.colorbar(label='Mean MSE')
plt.xticks(
    range(len(OPENING_RADII)),
    OPENING_RADII
)
plt.yticks(
    range(len(DILATION_RADII)),
    DILATION_RADII
)
plt.xlabel('Opening radius')
plt.ylabel('Dilation radius')
plt.title('Morphology parameter grid')
plt.tight_layout()
plt.show()



### 4. Step 4 vs Step 5

Step 4 uses the original morphology `(2, 2)`.

Step 5 uses the best measured pair while keeping every earlier improvement unchanged.


In [ ]:

print('STEP 4 — morphology = (2, 2)')
print(
    'Mean SAD :',
    np.mean(
        best_spectral_result['sad']
    )
)
print(
    'Mean MSE :',
    np.mean(
        best_spectral_result['mse']
    )
)
print(
    'Mean PSNR:',
    np.mean(
        best_spectral_result['psnr']
    )
)

print()

print(
    'STEP 5 — morphology =',
    BEST_MORPHOLOGY_PAIR
)
print(
    'Mean SAD :',
    np.mean(
        best_morphology_result['sad']
    )
)
print(
    'Mean MSE :',
    np.mean(
        best_morphology_result['mse']
    )
)
print(
    'Mean PSNR:',
    np.mean(
        best_morphology_result['psnr']
    )
)



### Step-5 Decision Rule

The best morphology is retained only if it improves the sequence-level metrics.

For very thin tools, smaller structuring elements can be preferable because aggressive dilation or opening can destroy geometric fidelity.


## 8. Optimize the Segmentation Threshold

The retained pipeline is now fixed at:

- temporal-median background;
- fixed pre-subtraction histogram mapping;
- Gaussian spatial filter `sigma = 0.75`;
- Gaussian high-pass spectral filter `D0 = 10`;
- morphology `(dilation radius = 1, opening radius = 0)`.

For **Threshold ablation**, only the segmentation threshold changes.

The original notebook uses:

```python
im = im > 0.1
```

The threshold is re-estimated on the retained feature representation rather than inherited from the baseline.



### 1. Why the Threshold Must Be Re-Optimized

The original threshold `0.1` was chosen for the original pipeline.

But after Improvements 1–5, the image presented to the segmentation stage is no longer identical:

```text
temporal-median background
+ fixed histogram mapping
+ sigma = 0.75
+ Gaussian HPF D0 = 10
+ dilation radius = 1
```

Therefore the old threshold should not be assumed optimal.

A threshold that is too low creates too many foreground detections.

A threshold that is too high misses thin guidewire / microcatheter pixels.

We first perform a coarse sweep, then a finer sweep around the best region.


In [ ]:

COARSE_THRESHOLDS = [
    0.02,
    0.04,
    0.06,
    0.08,
    0.10,
    0.12,
    0.14,
    0.16,
    0.20,
    0.25,
]

print(
    "Coarse threshold candidates:",
    COARSE_THRESHOLDS
)



### 2. Reusable Threshold Evaluation

Only this line changes:

```python
im = im > threshold
```

Every earlier processing step remains exactly as retained in Step 5.


In [ ]:

def evaluate_threshold(
    threshold_value
):
    errors = {
        'sad': [],
        'mse': [],
        'psnr': []
    }

    for num_f in frame_numbers:

        im_f = os.path.join(
            IMDIR,
            f"frame_{num_f}.png"
        )

        guidewire_f = os.path.join(
            IMDIR,
            f"{num_f}_GuideWire.tiff"
        )

        microcath_f = os.path.join(
            IMDIR,
            f"{num_f}_MicroCath.tiff"
        )

        image = io.imread(
            im_f,
            as_gray=True
        )

        guidewire_mask = io.imread(
            guidewire_f
        )

        if len(
            guidewire_mask.shape
        ) > 2:
            guidewire_mask = (
                guidewire_mask[:, :, 0]
            )

        microcath_mask = io.imread(
            microcath_f
        )

        if len(
            microcath_mask.shape
        ) > 2:
            microcath_mask = (
                microcath_mask[:, :, 0]
            )

        gt_mask = guidewire_mask.copy()

        gt_mask[
            np.where(
                microcath_mask < 128
            )
        ] = 0

        # Ground-truth thickening remains unchanged.
        gt_mask = erosion(
            gt_mask,
            disk(2)
        )

        image = (
            image.astype(float)
            / 255.
        )

        gt_mask = (
            gt_mask.astype(float)
            / 255.
        )

        # Step 3 retained.
        im = gaussian(
            image.copy(),
            sigma=BEST_SIGMA
        )

        # Step 2 retained.
        im = step3_fixed_map(
            im
        )

        # Step 1 retained.
        im = -(
            im
            - step3_background
        )

        # Step 4 retained.
        im, _, _ = (
            apply_spectral_high_pass(
                im,
                cutoff=BEST_SPECTRAL_CUTOFF
            )
        )

        # Step 5 retained:
        # best dilation radius = 1.
        if BEST_MORPHOLOGY_PAIR[0] > 0:
            im = dilation(
                im,
                disk(
                    BEST_MORPHOLOGY_PAIR[0]
                )
            )

        # Original second histogram transformation.
        im = saturate(
            im,
            10,
            100
        )

        # ONLY controlled variable:
        # segmentation threshold.
        im = (
            im > threshold_value
        )

        # Step 5 retained:
        # best opening radius = 0.
        if BEST_MORPHOLOGY_PAIR[1] > 0:
            im = opening(
                im,
                disk(
                    BEST_MORPHOLOGY_PAIR[1]
                )
            )

        im = 1 - im
        im_mask = im.copy()

        sad_i, mse_i, _ = (
            compute_errors(
                gt_mask.copy(),
                im_mask.copy()
            )
        )

        errors['sad'].append(
            sad_i
        )

        errors['mse'].append(
            mse_i
        )

        errors['psnr'].append(
            psnr(
                mse_i
            )
        )

    return errors



### 3. Coarse Threshold Sweep

The coarse sweep locates the useful threshold region.


In [ ]:

coarse_threshold_results = {}

for threshold_value in COARSE_THRESHOLDS:
    coarse_threshold_results[
        threshold_value
    ] = evaluate_threshold(
        threshold_value
    )

print(
    f"{'Threshold':>10s} "
    f"{'Mean SAD':>12s} "
    f"{'Mean MSE':>12s} "
    f"{'Mean PSNR':>12s}"
)

print(
    "-" * 51
)

for threshold_value in COARSE_THRESHOLDS:
    result = coarse_threshold_results[
        threshold_value
    ]

    print(
        f"{threshold_value:10.3f} "
        f"{np.mean(result['sad']):12.6f} "
        f"{np.mean(result['mse']):12.6f} "
        f"{np.mean(result['psnr']):12.6f}"
    )


In [ ]:

BEST_COARSE_THRESHOLD = min(
    COARSE_THRESHOLDS,
    key=lambda threshold_value:
        np.mean(
            coarse_threshold_results[
                threshold_value
            ][
                'mse'
            ]
        )
)

print(
    "Best coarse threshold:",
    BEST_COARSE_THRESHOLD
)



### 4. Fine Threshold Sweep

A finer search is performed around the best coarse value.

The search interval is clipped to valid positive thresholds.


In [ ]:

fine_start = max(
    0.005,
    BEST_COARSE_THRESHOLD - 0.03
)

fine_stop = (
    BEST_COARSE_THRESHOLD
    + 0.0301
)

FINE_THRESHOLDS = np.round(
    np.arange(
        fine_start,
        fine_stop,
        0.005
    ),
    3
).tolist()

print(
    "Fine threshold candidates:",
    FINE_THRESHOLDS
)


In [ ]:

fine_threshold_results = {}

for threshold_value in FINE_THRESHOLDS:
    fine_threshold_results[
        threshold_value
    ] = evaluate_threshold(
        threshold_value
    )

print(
    f"{'Threshold':>10s} "
    f"{'Mean SAD':>12s} "
    f"{'Mean MSE':>12s} "
    f"{'Mean PSNR':>12s}"
)

print(
    "-" * 51
)

for threshold_value in FINE_THRESHOLDS:
    result = fine_threshold_results[
        threshold_value
    ]

    print(
        f"{threshold_value:10.3f} "
        f"{np.mean(result['sad']):12.6f} "
        f"{np.mean(result['mse']):12.6f} "
        f"{np.mean(result['psnr']):12.6f}"
    )



### 5. Select the Best Threshold

The retained threshold is selected using the same rule as the previous steps:

- minimize mean MSE;
- confirm that mean SAD decreases;
- confirm that mean PSNR increases.


In [ ]:

BEST_THRESHOLD = min(
    FINE_THRESHOLDS,
    key=lambda threshold_value:
        np.mean(
            fine_threshold_results[
                threshold_value
            ][
                'mse'
            ]
        )
)

best_threshold_result = (
    fine_threshold_results[
        BEST_THRESHOLD
    ]
)

print(
    "Best threshold:",
    BEST_THRESHOLD
)

print(
    "Mean SAD :",
    np.mean(
        best_threshold_result[
            'sad'
        ]
    )
)

print(
    "Mean MSE :",
    np.mean(
        best_threshold_result[
            'mse'
        ]
    )
)

print(
    "Mean PSNR:",
    np.mean(
        best_threshold_result[
            'psnr'
        ]
    )
)



### 6. Threshold Sensitivity Curve

A good threshold should lie in a stable neighborhood rather than being an isolated numerical accident.

The curve below shows the fine-sweep behavior around the selected threshold.


In [ ]:

fine_mean_sad = [
    np.mean(
        fine_threshold_results[
            threshold_value
        ][
            'sad'
        ]
    )
    for threshold_value in FINE_THRESHOLDS
]

fine_mean_mse = [
    np.mean(
        fine_threshold_results[
            threshold_value
        ][
            'mse'
        ]
    )
    for threshold_value in FINE_THRESHOLDS
]

fine_mean_psnr = [
    np.mean(
        fine_threshold_results[
            threshold_value
        ][
            'psnr'
        ]
    )
    for threshold_value in FINE_THRESHOLDS
]

plt.figure(
    figsize=(12, 4)
)

plt.subplot(
    1,
    3,
    1
)

plt.plot(
    FINE_THRESHOLDS,
    fine_mean_sad,
    marker='o'
)

plt.axvline(
    BEST_THRESHOLD,
    linestyle='--'
)

plt.title(
    'Mean SAD vs threshold'
)

plt.xlabel(
    'Threshold'
)

plt.subplot(
    1,
    3,
    2
)

plt.plot(
    FINE_THRESHOLDS,
    fine_mean_mse,
    marker='o'
)

plt.axvline(
    BEST_THRESHOLD,
    linestyle='--'
)

plt.title(
    'Mean MSE vs threshold'
)

plt.xlabel(
    'Threshold'
)

plt.subplot(
    1,
    3,
    3
)

plt.plot(
    FINE_THRESHOLDS,
    fine_mean_psnr,
    marker='o'
)

plt.axvline(
    BEST_THRESHOLD,
    linestyle='--'
)

plt.title(
    'Mean PSNR vs threshold'
)

plt.xlabel(
    'Threshold'
)

plt.tight_layout()
plt.show()



### Baseline Threshold vs Re-optimized Threshold

Step 5 uses the original segmentation threshold:

```text
threshold = 0.1
```

Step 6 uses the best measured threshold while keeping all earlier improvements fixed.


In [ ]:

print(
    'STEP 5 — threshold = 0.1'
)

print(
    'Mean SAD :',
    np.mean(
        best_morphology_result[
            'sad'
        ]
    )
)

print(
    'Mean MSE :',
    np.mean(
        best_morphology_result[
            'mse'
        ]
    )
)

print(
    'Mean PSNR:',
    np.mean(
        best_morphology_result[
            'psnr'
        ]
    )
)

print()

print(
    f'STEP 6 — threshold = {BEST_THRESHOLD}'
)

print(
    'Mean SAD :',
    np.mean(
        best_threshold_result[
            'sad'
        ]
    )
)

print(
    'Mean MSE :',
    np.mean(
        best_threshold_result[
            'mse'
        ]
    )
)

print(
    'Mean PSNR:',
    np.mean(
        best_threshold_result[
            'psnr'
        ]
    )
)



### Step-6 Decision Rule

The threshold is retained only if it improves the sequence-level result.

This step completes the controlled optimization of the **classical threshold-based branch**.

The next logical stage is no longer another threshold parameter. It is to evaluate the second segmentation strategy present in the original notebook:

> **Expectation Maximization / Gaussian Mixture Model**

That comparison should be performed only after the classical branch is fully optimized.



### Step-6 Validated Result

Independent sequence-level validation with the retained Step-5 pipeline gives:

| Threshold | Mean SAD | Mean MSE | Mean PSNR |
|---:|---:|---:|---:|
| 0.100 | 1.363209 | 347.618237 | 23.193967 dB |
| **0.105** | **1.358442** | **346.402788** | **23.266125 dB** |

The retained threshold is therefore:

\[
\boxed{T = 0.105}
\]

The improvement is deliberately small. This is still useful because it shows that the original value `0.1` was already close to the optimum after the previous processing improvements.

We stop the threshold search here rather than tuning to increasingly tiny increments on the same sequence.

The next controlled experiment is the second segmentation branch already present in the original notebook:

> **Expectation Maximization / Gaussian Mixture Model (GMM)**.


## 9. Compare Threshold Segmentation with EM/GMM

The original laboratory notebook contains a second segmentation branch based on:

> **Expectation Maximization (EM) with a Gaussian Mixture Model (GMM)**

The EM/GMM branch is evaluated under the same retained preprocessing configuration as the deterministic threshold branch.

The retained preprocessing remains:

- temporal-median background;
- fixed pre-subtraction histogram mapping;
- Gaussian spatial filter `sigma = 0.75`;
- Gaussian spectral high-pass filter `D0 = 10`;
- dilation radius `1`;
- opening radius `0`.

Only the segmentation decision changes.

### Deterministic Threshold Branch

```python
tool_mask = feature > 0.105
```

### EM/GMM Branch

```python
gmm = GaussianMixture(n_components=2)
labels = gmm.fit_predict(...)
```

The GMM therefore replaces **only the thresholding stage**.



### 1. EM / GMM Intuition

A Gaussian Mixture Model represents a probability density as a weighted sum of Gaussian components:

\[
p(x)
=
\sum_{k=1}^{K}
\pi_k
\mathcal{N}
\left(
x \mid \mu_k,\Sigma_k
\right)
\]

For this binary segmentation experiment we use:

\[
K=2
\]

with the interpretation:

- one Gaussian component = low-response/background-like pixels;
- one Gaussian component = high-response/tool-like pixels.

The EM algorithm alternates between:

### E-step

Estimate the probability that each sample belongs to each Gaussian component.

### M-step

Update the Gaussian parameters using those probabilities.

The iterations continue until convergence.



### 2. Why Two Components?

The segmentation objective is binary:

```text
background
vs
moving tool
```

Therefore two components provide the most direct probabilistic analogue of the Step-6 binary threshold.

The component with the **largest mean feature response** is interpreted as the moving-tool component.



### 3. GMM Segmentation Function

The feature image contains \(512\times512=262,144\) pixels.

To keep fitting efficient:

- model parameters are fitted on a deterministic random sample of at most `50,000` pixels;
- after fitting, the model predicts a component label for **every pixel**.

The random seed is fixed to make the experiment reproducible.


In [ ]:

GMM_FIT_SAMPLE_SIZE = 50_000
GMM_RANDOM_STATE = 0


def gmm_segment(
    feature,
    fit_sample_size=GMM_FIT_SAMPLE_SIZE
):
    X = feature.reshape(
        -1,
        1
    ).astype(
        np.float64
    )

    rng = np.random.default_rng(
        GMM_RANDOM_STATE
    )

    sample_size = min(
        fit_sample_size,
        X.shape[0]
    )

    sample_indices = rng.choice(
        X.shape[0],
        size=sample_size,
        replace=False
    )

    gmm = GaussianMixture(
        n_components=2,
        covariance_type='full',
        random_state=GMM_RANDOM_STATE,
        max_iter=200,
        n_init=3
    )

    gmm.fit(
        X[
            sample_indices
        ]
    )

    labels = gmm.predict(
        X
    ).reshape(
        feature.shape
    )

    component_means = (
        gmm.means_.ravel()
    )

    tool_component = int(
        np.argmax(
            component_means
        )
    )

    tool_mask = (
        labels
        == tool_component
    )

    return (
        tool_mask,
        gmm
    )



### 4. Rebuild the Retained Step-6 Feature Image

This helper reproduces the exact Step-6 preprocessing **up to the segmentation decision**.

No threshold is applied inside this function.


In [ ]:

def build_step6_feature(
    num_f
):
    image = io.imread(
        os.path.join(
            IMDIR,
            f"frame_{num_f}.png"
        ),
        as_gray=True
    )

    image = (
        image.astype(float)
        / 255.
    )

    # Step 3 retained.
    im = gaussian(
        image.copy(),
        sigma=BEST_SIGMA
    )

    # Step 2 retained.
    im = step3_fixed_map(
        im
    )

    # Step 1 retained.
    im = -(
        im
        - step3_background
    )

    # Step 4 retained.
    im, _, _ = (
        apply_spectral_high_pass(
            im,
            cutoff=BEST_SPECTRAL_CUTOFF
        )
    )

    # Step 5 retained:
    # dilation radius = 1.
    if BEST_MORPHOLOGY_PAIR[0] > 0:
        im = dilation(
            im,
            disk(
                BEST_MORPHOLOGY_PAIR[0]
            )
        )

    # Original second saturation retained.
    im = saturate(
        im,
        10,
        100
    )

    return im



### 5. Apply the GMM to the Entire Sequence

For every frame:

1. build the exact Step-6 feature image;
2. fit a two-component GMM;
3. identify the higher-mean component as tool;
4. convert to the original TP mask convention:
   - `0 = tool`
   - `1 = background`;
5. compute the same SAD / MSE / PSNR metrics.


In [ ]:

gmm_error_dic = {
    'sad': [],
    'mse': [],
    'psnr': []
}

gmm_component_means = {}
gmm_masks = {}

for num_f in frame_numbers:

    feature = build_step6_feature(
        num_f
    )

    tool_mask, gmm_model = (
        gmm_segment(
            feature
        )
    )

    # Step-5 opening radius is 0,
    # so no additional opening is applied.

    im_mask = (
        1
        - tool_mask.astype(float)
    )

    # Load ground truth exactly as before.
    guidewire_mask = io.imread(
        os.path.join(
            IMDIR,
            f"{num_f}_GuideWire.tiff"
        )
    )

    if len(
        guidewire_mask.shape
    ) > 2:
        guidewire_mask = (
            guidewire_mask[:, :, 0]
        )

    microcath_mask = io.imread(
        os.path.join(
            IMDIR,
            f"{num_f}_MicroCath.tiff"
        )
    )

    if len(
        microcath_mask.shape
    ) > 2:
        microcath_mask = (
            microcath_mask[:, :, 0]
        )

    gt_mask = guidewire_mask.copy()

    gt_mask[
        np.where(
            microcath_mask < 128
        )
    ] = 0

    gt_mask = erosion(
        gt_mask,
        disk(2)
    )

    gt_mask = (
        gt_mask.astype(float)
        / 255.
    )

    sad_i, mse_i, _ = (
        compute_errors(
            gt_mask.copy(),
            im_mask.copy()
        )
    )

    gmm_error_dic[
        'sad'
    ].append(
        sad_i
    )

    gmm_error_dic[
        'mse'
    ].append(
        mse_i
    )

    gmm_error_dic[
        'psnr'
    ].append(
        psnr(
            mse_i
        )
    )

    gmm_component_means[
        num_f
    ] = np.sort(
        gmm_model.means_.ravel()
    )

    gmm_masks[
        num_f
    ] = im_mask


print(
    f"{'Frame':>6s} "
    f"{'Low mean':>12s} "
    f"{'High mean':>12s}"
)

print(
    "-" * 34
)

for num_f in frame_numbers:
    means = (
        gmm_component_means[
            num_f
        ]
    )

    print(
        f"{num_f:6d} "
        f"{means[0]:12.6f} "
        f"{means[1]:12.6f}"
    )



### 6. Threshold vs GMM — Quantitative Comparison

The only difference is now:

```text
Step 6 → deterministic threshold at 0.105
Step 7 → probabilistic two-component GMM
```

Everything before segmentation is identical.


In [ ]:

print(
    'STEP 6 — THRESHOLD = 0.105'
)

print(
    'Mean SAD :',
    np.mean(
        best_threshold_result[
            'sad'
        ]
    )
)

print(
    'Mean MSE :',
    np.mean(
        best_threshold_result[
            'mse'
        ]
    )
)

print(
    'Mean PSNR:',
    np.mean(
        best_threshold_result[
            'psnr'
        ]
    )
)

print()

print(
    'STEP 7 — GMM, 2 COMPONENTS'
)

print(
    'Mean SAD :',
    np.mean(
        gmm_error_dic[
            'sad'
        ]
    )
)

print(
    'Mean MSE :',
    np.mean(
        gmm_error_dic[
            'mse'
        ]
    )
)

print(
    'Mean PSNR:',
    np.mean(
        gmm_error_dic[
            'psnr'
        ]
    )
)



### 7. Per-Frame Comparison

A method can have a reasonable mean while failing badly on specific frames.

Therefore the comparison is also shown frame by frame.


In [ ]:

step6_sad = np.array(
    best_threshold_result[
        'sad'
    ]
)

step6_mse = np.array(
    best_threshold_result[
        'mse'
    ]
)

step6_psnr = np.array(
    best_threshold_result[
        'psnr'
    ]
)

step7_sad = np.array(
    gmm_error_dic[
        'sad'
    ]
)

step7_mse = np.array(
    gmm_error_dic[
        'mse'
    ]
)

step7_psnr = np.array(
    gmm_error_dic[
        'psnr'
    ]
)

plt.figure(
    figsize=(12, 4)
)

plt.subplot(
    1,
    3,
    1
)

plt.plot(
    frame_numbers,
    step6_sad,
    marker='o',
    label='Threshold 0.105'
)

plt.plot(
    frame_numbers,
    step7_sad,
    marker='o',
    label='GMM'
)

plt.title(
    'SAD'
)

plt.xlabel(
    'Frame'
)

plt.legend()

plt.subplot(
    1,
    3,
    2
)

plt.plot(
    frame_numbers,
    step6_mse,
    marker='o',
    label='Threshold 0.105'
)

plt.plot(
    frame_numbers,
    step7_mse,
    marker='o',
    label='GMM'
)

plt.title(
    'MSE'
)

plt.xlabel(
    'Frame'
)

plt.legend()

plt.subplot(
    1,
    3,
    3
)

plt.plot(
    frame_numbers,
    step6_psnr,
    marker='o',
    label='Threshold 0.105'
)

plt.plot(
    frame_numbers,
    step7_psnr,
    marker='o',
    label='GMM'
)

plt.title(
    'PSNR'
)

plt.xlabel(
    'Frame'
)

plt.legend()

plt.tight_layout()
plt.show()



### 8. Representative Visual Comparison

The threshold and GMM masks are compared on a representative frame.

Remember:

- black = detected tool;
- white = background.


In [ ]:

REPRESENTATIVE_GMM_FRAME = 251

feature_rep = build_step6_feature(
    REPRESENTATIVE_GMM_FRAME
)

threshold_tool = (
    feature_rep
    > BEST_THRESHOLD
)

threshold_mask = (
    1
    - threshold_tool.astype(float)
)

gmm_mask_rep = (
    gmm_masks[
        REPRESENTATIVE_GMM_FRAME
    ]
)

plt.figure(
    figsize=(15, 5)
)

plt.subplot(
    1,
    3,
    1
)

plt.imshow(
    feature_rep,
    cmap='gray'
)

plt.title(
    'Step-6 feature image'
)

plt.axis(
    'off'
)

plt.subplot(
    1,
    3,
    2
)

plt.imshow(
    threshold_mask,
    cmap='gray',
    vmin=0,
    vmax=1
)

plt.title(
    'Threshold = 0.105'
)

plt.axis(
    'off'
)

plt.subplot(
    1,
    3,
    3
)

plt.imshow(
    gmm_mask_rep,
    cmap='gray',
    vmin=0,
    vmax=1
)

plt.title(
    '2-component GMM'
)

plt.axis(
    'off'
)

plt.tight_layout()
plt.show()



### 9. Validated Step-7 Result

On the full ten-frame sequence, the validated comparison is:

| Segmentation | Mean SAD | Mean MSE | Mean PSNR |
|---|---:|---:|---:|
| **Threshold `0.105`** | **1.358442** | **346.402788** | **23.266125 dB** |
| GMM, 2 components | 2.468250 | 629.403820 | 20.461009 dB |

Therefore the GMM branch is **not retained**.

The result is important because GMM is more sophisticated statistically, but sophistication alone does not guarantee better segmentation.



### Why Does the GMM Perform Worse Here?

The feature distribution is highly imbalanced:

- background pixels dominate the image;
- tool pixels represent only a very small fraction.

A two-component GMM attempts to model the global intensity distribution.

Its second Gaussian therefore does not necessarily correspond exactly to the sparse moving-tool class.

It can absorb:

- anatomical residual edges;
- acquisition noise;
- other high-response pixels.

The optimized threshold, in contrast, directly selects only the strongest feature responses and performs better on this specific sequence.



### Step-7 Decision

### Tested

- EM / Gaussian Mixture Model;
- 2 components;
- same Step-6 preprocessing;
- same ground truth;
- same SAD / MSE / PSNR metrics.

### Result

**Rejected as final segmentation method.**

### Retained segmentation

\[
\boxed{T = 0.105}
\]

This completes the controlled comparison between the two segmentation strategies present in the original notebook.

The next improvement should return to the retained threshold branch rather than forcing GMM into the final pipeline.


## 10. Restrict Processing to a Valid Field-of-View Mask

The retained threshold-based pipeline remains:

- temporal-median background;
- fixed pre-subtraction histogram mapping;
- Gaussian spatial filter `sigma = 0.75`;
- Gaussian spectral high-pass filter `D0 = 10`;
- dilation radius `1`;
- opening radius `0`;
- segmentation threshold `0.105`.

The GMM branch was evaluated and rejected on the measured sequence-level evidence.

For **Field-of-view ablation**, only one new operation is added:

> a fixed field-of-view (FOV) mask derived from the temporal-median background.

The purpose is to remove detections outside the useful anatomical region while keeping all true guidewire / microcatheter pixels.



### 1. FOV-Mask Principle

The median background contains a stable representation of the anatomy and the bright external image regions.

A simple FOV candidate is built by:

1. thresholding the median background;
2. keeping the largest connected dark region;
3. filling internal holes.

For threshold \(T_{FOV}\):

\[
R(x,y)
=
B_{\text{median}}(x,y)
<
T_{FOV}
\]

We then keep the largest connected component of \(R\).

The resulting mask is fixed once and applied identically to every frame.


In [ ]:

from scipy import ndimage



### 2. Build the Raw Median Image

The FOV is derived from the raw temporal median rather than from the contrast-normalized residual.

That keeps the ROI definition tied to stable anatomy / image support rather than to moving-tool responses.


In [ ]:

raw_sequence = []

for num_f in frame_numbers:
    image = io.imread(
        os.path.join(
            IMDIR,
            f"frame_{num_f}.png"
        ),
        as_gray=True
    )

    # Convert to the original 0–255 intensity domain
    # for an interpretable FOV threshold.
    if image.max() <= 1.0:
        image_raw = (
            image * 255.0
        )
    else:
        image_raw = image.astype(float)

    raw_sequence.append(
        image_raw
    )

raw_sequence = np.stack(
    raw_sequence,
    axis=0
)

raw_median_background = np.median(
    raw_sequence,
    axis=0
)

print(
    "Raw median range:",
    raw_median_background.min(),
    "to",
    raw_median_background.max()
)



### 3. Reusable FOV Builder

Only the **largest connected region** is retained.

This prevents isolated dark patches outside the useful image region from being accepted as valid FOV.


In [ ]:

def build_fov_mask(
    background,
    intensity_threshold
):
    dark_region = (
        background
        < intensity_threshold
    )

    labels, num_labels = (
        ndimage.label(
            dark_region
        )
    )

    if num_labels == 0:
        raise ValueError(
            "No connected FOV region found."
        )

    component_sizes = (
        ndimage.sum(
            dark_region,
            labels,
            range(
                1,
                num_labels + 1
            )
        )
    )

    largest_label = (
        int(
            np.argmax(
                component_sizes
            )
        )
        + 1
    )

    largest_component = (
        labels
        == largest_label
    )

    fov_mask = (
        ndimage.binary_fill_holes(
            largest_component
        )
    )

    return fov_mask



### 4. Candidate FOV Thresholds

The threshold must satisfy two competing objectives:

- remove as much irrelevant exterior region as possible;
- retain the complete annotated tool trajectory.

We therefore evaluate both:

- segmentation metrics;
- ground-truth tool coverage inside the ROI.


In [ ]:

FOV_THRESHOLDS = [
    120,
    125,
    130,
    135,
    140,
    150,
    160,
    180,
    200,
]

print(
    "FOV thresholds:",
    FOV_THRESHOLDS
)



### 5. Rebuild the Retained Step-6 Threshold Prediction

Step 7 tested GMM, but GMM was rejected.

The final branch therefore returns to the retained Step-6 threshold decision:

\[
T = 0.105
\]

The FOV mask is applied **after segmentation**:

\[
T_{\text{ROI}}
=
T_{\text{pred}}
\cap
R_{FOV}
\]

No other preprocessing parameter changes.


In [ ]:

def retained_threshold_tool_mask(
    num_f
):
    feature = build_step6_feature(
        num_f
    )

    return (
        feature
        > BEST_THRESHOLD
    )



### 6. Ground-Truth Coverage and Metric Evaluation

For each FOV threshold we measure:

- fraction of the image retained;
- minimum fraction of annotated tool pixels retained across all 10 frames;
- mean SAD;
- mean MSE;
- mean PSNR.

A candidate that removes true tool pixels is not acceptable even if the global pixel metric improves.


In [ ]:

def evaluate_fov_threshold(
    fov_threshold
):
    fov_mask = build_fov_mask(
        raw_median_background,
        fov_threshold
    )

    errors = {
        'sad': [],
        'mse': [],
        'psnr': []
    }

    gt_coverages = []

    for num_f in frame_numbers:

        pred_tool = (
            retained_threshold_tool_mask(
                num_f
            )
        )

        pred_tool_roi = (
            pred_tool
            & fov_mask
        )

        pred_mask = (
            1
            - pred_tool_roi.astype(float)
        )

        guidewire_mask = io.imread(
            os.path.join(
                IMDIR,
                f"{num_f}_GuideWire.tiff"
            )
        )

        if len(
            guidewire_mask.shape
        ) > 2:
            guidewire_mask = (
                guidewire_mask[:, :, 0]
            )

        microcath_mask = io.imread(
            os.path.join(
                IMDIR,
                f"{num_f}_MicroCath.tiff"
            )
        )

        if len(
            microcath_mask.shape
        ) > 2:
            microcath_mask = (
                microcath_mask[:, :, 0]
            )

        # Raw tool-positive GT for ROI-coverage check.
        gt_tool_raw = (
            (guidewire_mask < 128)
            | (microcath_mask < 128)
        )

        gt_coverage = (
            np.sum(
                gt_tool_raw
                & fov_mask
            )
            / (
                np.sum(
                    gt_tool_raw
                )
                + 1e-12
            )
        )

        gt_coverages.append(
            gt_coverage
        )

        # Same official thickened GT used by all earlier steps.
        gt_mask = (
            guidewire_mask.copy()
        )

        gt_mask[
            np.where(
                microcath_mask < 128
            )
        ] = 0

        gt_mask = erosion(
            gt_mask,
            disk(2)
        )

        gt_mask = (
            gt_mask.astype(float)
            / 255.
        )

        sad_i, mse_i, _ = (
            compute_errors(
                gt_mask.copy(),
                pred_mask.copy()
            )
        )

        errors['sad'].append(
            sad_i
        )

        errors['mse'].append(
            mse_i
        )

        errors['psnr'].append(
            psnr(
                mse_i
            )
        )

    return {
        'fov_mask':
            fov_mask,
        'area_fraction':
            float(
                np.mean(
                    fov_mask
                )
            ),
        'min_gt_coverage':
            float(
                np.min(
                    gt_coverages
                )
            ),
        'mean_gt_coverage':
            float(
                np.mean(
                    gt_coverages
                )
            ),
        'sad':
            errors['sad'],
        'mse':
            errors['mse'],
        'psnr':
            errors['psnr'],
    }


In [ ]:

fov_results = {}

for fov_threshold in FOV_THRESHOLDS:
    fov_results[
        fov_threshold
    ] = evaluate_fov_threshold(
        fov_threshold
    )

print(
    f"{'FOV T':>7s} "
    f"{'Area':>8s} "
    f"{'Min GT':>9s} "
    f"{'Mean SAD':>12s} "
    f"{'Mean MSE':>12s} "
    f"{'Mean PSNR':>12s}"
)

print(
    "-" * 70
)

for fov_threshold in FOV_THRESHOLDS:
    result = fov_results[
        fov_threshold
    ]

    print(
        f"{fov_threshold:7d} "
        f"{result['area_fraction']:8.4f} "
        f"{result['min_gt_coverage']:9.4f} "
        f"{np.mean(result['sad']):12.6f} "
        f"{np.mean(result['mse']):12.6f} "
        f"{np.mean(result['psnr']):12.6f}"
    )



### 7. Select a Safe FOV Threshold

A candidate is considered **safe** only if all annotated raw tool pixels are retained:

\[
\min_t
\mathrm{Coverage}_t
=
1.0
\]

Among safe candidates, we select the one with the lowest mean MSE.

This prevents the ROI from improving the metric simply by deleting parts of the true tool.


In [ ]:

safe_fov_thresholds = [
    threshold
    for threshold in FOV_THRESHOLDS
    if (
        fov_results[
            threshold
        ][
            'min_gt_coverage'
        ]
        >= 0.999999
    )
]

BEST_FOV_THRESHOLD = min(
    safe_fov_thresholds,
    key=lambda threshold:
        np.mean(
            fov_results[
                threshold
            ][
                'mse'
            ]
        )
)

best_fov_result = (
    fov_results[
        BEST_FOV_THRESHOLD
    ]
)

BEST_FOV_MASK = (
    best_fov_result[
        'fov_mask'
    ]
)

print(
    "Best safe FOV threshold:",
    BEST_FOV_THRESHOLD
)

print(
    "FOV area fraction:",
    best_fov_result[
        'area_fraction'
    ]
)

print(
    "Minimum GT coverage:",
    best_fov_result[
        'min_gt_coverage'
    ]
)

print(
    "Mean SAD :",
    np.mean(
        best_fov_result[
            'sad'
        ]
    )
)

print(
    "Mean MSE :",
    np.mean(
        best_fov_result[
            'mse'
        ]
    )
)

print(
    "Mean PSNR:",
    np.mean(
        best_fov_result[
            'psnr'
        ]
    )
)



### 8. Visualize the FOV

The retained FOV should remove mainly external bright regions while preserving the complete tool trajectory.


In [ ]:

plt.figure(
    figsize=(12, 4)
)

plt.subplot(
    1,
    3,
    1
)

plt.imshow(
    raw_median_background,
    cmap='gray'
)

plt.title(
    'Raw temporal median'
)

plt.axis(
    'off'
)

plt.subplot(
    1,
    3,
    2
)

plt.imshow(
    BEST_FOV_MASK,
    cmap='gray'
)

plt.title(
    f'FOV mask — T={BEST_FOV_THRESHOLD}'
)

plt.axis(
    'off'
)

plt.subplot(
    1,
    3,
    3
)

masked_background = (
    raw_median_background.copy()
)

masked_background[
    ~BEST_FOV_MASK
] = 255

plt.imshow(
    masked_background,
    cmap='gray'
)

plt.title(
    'Retained field of view'
)

plt.axis(
    'off'
)

plt.tight_layout()
plt.show()



### 9. Step 6 vs Step 8

Step 7 was a comparison experiment and GMM was rejected.

Therefore the correct baseline for this new improvement is the retained threshold branch from Step 6.

The only difference in Step 8 is the fixed FOV mask.


In [ ]:

print(
    'RETAINED THRESHOLD PIPELINE — NO FOV'
)

print(
    'Mean SAD :',
    np.mean(
        best_threshold_result[
            'sad'
        ]
    )
)

print(
    'Mean MSE :',
    np.mean(
        best_threshold_result[
            'mse'
        ]
    )
)

print(
    'Mean PSNR:',
    np.mean(
        best_threshold_result[
            'psnr'
        ]
    )
)

print()

print(
    'STEP 8 — FOV THRESHOLD =',
    BEST_FOV_THRESHOLD
)

print(
    'Mean SAD :',
    np.mean(
        best_fov_result[
            'sad'
        ]
    )
)

print(
    'Mean MSE :',
    np.mean(
        best_fov_result[
            'mse'
        ]
    )
)

print(
    'Mean PSNR:',
    np.mean(
        best_fov_result[
            'psnr'
        ]
    )
)



### 10. Validated Step-8 Result

Independent validation on the ten supplied frames gave:

| Version | Mean SAD | Mean MSE | Mean PSNR |
|---|---:|---:|---:|
| Threshold pipeline, no FOV | 1.358442 | 346.402788 | 23.266125 dB |
| **FOV threshold `130`** | **1.350174** | **344.294357** | **23.302722 dB** |

For `T_FOV = 130`:

- approximately **81.7%** of the image is retained;
- **100% of raw annotated tool pixels** remain inside the FOV on all ten frames.

The gain is small but consistent.



### Step-8 Decision

The FOV mask is **retained**, with:

\[
\boxed{T_{FOV}=130}
\]

However, its effect is deliberately described as modest.

It is useful because it:

- removes irrelevant external image regions;
- does not remove any annotated tool pixels in this sequence;
- slightly improves SAD / MSE / PSNR;
- gives the later pipeline a physically meaningful anatomical support.

The next improvements should continue from the retained **threshold + FOV** branch, not from the rejected GMM branch.


## 11. Assemble the Final Retained Pipeline

The retained branch is assembled explicitly from the decisions supported by Tasks 3–10. Rejected alternatives remain documented but are not silently included.

In [ ]:
FINAL_PIPELINE = {
    "background_model": "temporal median",
    "histogram_mapping": "fixed background-derived mapping",
    "spatial_sigma": float(BEST_SIGMA),
    "spectral_high_pass_cutoff": float(BEST_SPECTRAL_CUTOFF),
    "morphology": tuple(BEST_MORPHOLOGY_PAIR),
    "segmentation": "deterministic threshold",
    "threshold": float(BEST_THRESHOLD),
    "fov_threshold": int(BEST_FOV_THRESHOLD),
    "mask_convention": "0=tool, 1=background",
}

for key, value in FINAL_PIPELINE.items():
    print(f"{key}: {value}")


## 12. Compute Sequence-Level Quantitative Evaluation

Summarize the retained FOV-protected threshold pipeline over the complete ten-frame sequence.

In [ ]:
final_sad = np.asarray(best_fov_result["sad"], dtype=float)
final_mse = np.asarray(best_fov_result["mse"], dtype=float)
final_psnr = np.asarray(best_fov_result["psnr"], dtype=float)

print("Final sequence-level metrics")
print("Mean SAD :", final_sad.mean(), "+/-", final_sad.std())
print("Mean MSE :", final_mse.mean(), "+/-", final_mse.std())
print("Mean PSNR:", final_psnr.mean(), "+/-", final_psnr.std())


## 13. Run Numerical and Output-file Validation Checks

Run explicit numerical, logical, and output-file checks. This is the completion gate for the executable notebook.

In [ ]:
expected_figures = [
    "01_representative_data.png",
    "02_background_models.png",
    "03_histogram_transformation.png",
    "04_background_residual.png",
    "05_spatial_filtering.png",
    "06_spatial_sigma_sensitivity.png",
    "07_spectral_filtering.png",
    "08_segmentation.png",
    "09_threshold_sensitivity.png",
    "10_morphological_refinement.png",
    "11_mask_vs_ground_truth.png",
    "12_guidance_overlay.png",
    "13_validation_overlay.png",
    "14_sad_vs_time.png",
    "15_mse_vs_time.png",
    "16_psnr_vs_time.png",
    "17_overlap_vs_time.png",
]

assert len(frame_numbers) == 10
assert final_sad.shape == (10,)
assert final_mse.shape == (10,)
assert final_psnr.shape == (10,)
assert np.all(np.isfinite(final_sad))
assert np.all(np.isfinite(final_mse))
assert np.all(np.isfinite(final_psnr))
assert BEST_SIGMA > 0
assert BEST_SPECTRAL_CUTOFF > 0
assert 0 < BEST_THRESHOLD < 1
assert best_fov_result["min_gt_coverage"] >= 0.999999

output_dir = Path("../outputs/figures")
missing_outputs = [name for name in expected_figures if not (output_dir / name).exists()]
assert not missing_outputs, f"Missing required figures: {missing_outputs}"

print("All Background Subtraction validation checks passed.")


## Final Result Summary

The project preserves the original laboratory baseline and then applies controlled ablation to the background model, intensity mapping, spatial filtering, spectral filtering, morphology, segmentation, and field-of-view restriction. The final retained branch uses the threshold-based segmentation rather than the rejected GMM alternative and is evaluated with sequence-level SAD, MSE, PSNR, ground-truth coverage, and qualitative overlays.